# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. 

## Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and explore the records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata  # DatasetMetadata object

print(f"{metadata.name}: {metadata.description}")
print(f"Published on: {metadata.date_published}")
print(f"Keywords: {metadata.keywords}\n")

## 2. Data Overview

Examine available record sets and their fields. We reference **all entities by their `@id` fields** as required.

Let's list all record sets in the dataset and for each, show their fields and columns, with `@id` values.

In [ ]:
# List all record sets and their fields/columns using `@id` references

record_sets = dataset.metadata.record_sets
print(f"Total record sets: {len(record_sets)}")
for i, rs in enumerate(record_sets):
    print(f"[{i}] RecordSet @id: {rs.id}\n       name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
    print(f"    Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    # List fields (for tabular data)
    if hasattr(rs, 'fields') and rs.fields:
        print("    Fields (with @id):")
        for f in rs.fields:
            print(f"      - {f.id} ({f.name if hasattr(f, 'name') else 'N/A'})  datatype: {f.data_type if hasattr(f, 'data_type') else 'N/A'}")
    if hasattr(rs, 'columns') and rs.columns:
        print("    Columns (with @id):")
        for c in rs.columns:
            print(f"      - {c.id}  ({c.name if hasattr(c, 'name') else 'N/A'})  datatype: {getattr(c, 'data_type', 'N/A')}")
    print("\n")

## 3. Data Extraction

We will load data for the main record set, referencing by its `@id`. The largest record set typically contains the main tabular data. 

**Choose the record set and field ids as seen above.**

In [ ]:
# Extract data for all record sets, referencing them by @id
from collections import OrderedDict

# Select all record set ids detected in previous cell
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print("No records found.\n")

## 4. Exploratory Data Analysis (EDA)

We will demonstrate filtering based on a numeric field, normalization, and grouping. 

*Update `numeric_field_id` and `group_field_id` with values appropriate for your dataset from above.*

In [ ]:
# Example EDA for the main tabular record set

main_record_set_id = record_set_ids[0]  # or manually set as found above
df = dataframes[main_record_set_id]

# Identify a numeric field (inspect columns from previous step)
# Example: 'coverage_percent' (replace with real @id or column name from this dataset)
numeric_field_id = None
for col in df.columns:
    # Try to pick a likely numeric field
    if "coverage" in col.lower() or "count" in col.lower() or "abundance" in col.lower() or "mw" in col.lower():
        numeric_field_id = col
        break

if numeric_field_id:
    print(f"Using numeric field: {numeric_field_id}\n")
    # Remove missing values for the demo
    filt = pd.to_numeric(df[numeric_field_id], errors='coerce').notnull()
    num_df = df[filt].copy()
    num_df[numeric_field_id] = pd.to_numeric(num_df[numeric_field_id], errors='coerce')

    # Filter values > 10
    threshold = 10
    filtered_df = num_df[num_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a text field, e.g. 'description', 'sample', 'accession', etc.
    group_field_id = None
    for col in df.columns:
        if col != numeric_field_id and df[col].dtype == object and not pd.api.types.is_numeric_dtype(df[col]):
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by '{group_field_id}':")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA in this dataset.")

## 5. Visualization

Visualize the distribution of a numeric variable and the result of the grouping using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(num_df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group (if group_field_id)
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field found for plotting.")

## 6. Conclusion

In this notebook, we showcased how to load, inspect, and analyze a mass spectrometry dataset using the `mlcroissant` library. All data entities are referenced using their `@id` fields as recommended. This approach enables robust and FAIR data science workflows on complex scientific datasets.

Further analysis can include deeper protein-level, modification, or expression studies, enabled by the standardized Croissant schema and programmatic data acquisition.